In [1]:
!pip install scikit-learn==1.8.0
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/MINI_Project_BATCH8/Datasets/kidney_donor_recipient_large_dataset.csv',low_memory=False)

In [4]:
df.head()

,Donor_ID,Donor_Age,Donor_Gender,Donor_Blood_Type,Donor_HLA_A,Donor_HLA_B,Donor_HLA_DR,Donor_Creatinine_Level,Donor_BMI,Donor_Medical_History,...,Recipient_Blood_Type,Recipient_HLA_A,Recipient_HLA_B,Recipient_HLA_DR,Recipient_Creatinine_Level,Recipient_BMI,Recipient_Urgency_Level,Recipient_Medical_History,Compatibility_Score,Match_Label
0,D1,50,Male,A,A2,B8,DR7,1.44,30.0,Hypertension,...,A,A24,B7,DR17,4.03,31.9,Low,Diabetes,0.65,1
1,D2501,33,Female,B,A3,B7,DR4,0.56,26.0,NaN,...,A,A1,B27,DR7,3.45,20.0,Low,NaN,0.35,0
2,D2,36,Female,B,A3,B27,DR17,1.12,26.1,NaN,...,B,A1,B27,DR15,9.62,30.5,Low,NaN,0.83,1
3,D2502,32,Male,A,A1,B7,DR7,1.10,24.7,NaN,...,B,A3,B8,DR15,4.94,24.1,Low,Hypertension,0.44,0
4,D3,24,Male,A,A3,B44,DR15,1.20,24.1,NaN,...,A,A3,B7,DR7,9.73,34.1,High,NaN,0.82,1


In [5]:
df['Recipient_Medical_History'].unique()

array(['Diabetes', nan, 'Hypertension', 'Both'], dtype=object)

In [6]:
y = df["Match_Label"].astype(int)
X = df.drop(["Match_Label"], axis=1)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Donor_ID                    5000 non-null   object 
 1   Donor_Age                   5000 non-null   int64  
 2   Donor_Gender                5000 non-null   object 
 3   Donor_Blood_Type            5000 non-null   object 
 4   Donor_HLA_A                 5000 non-null   object 
 5   Donor_HLA_B                 5000 non-null   object 
 6   Donor_HLA_DR                5000 non-null   object 
 7   Donor_Creatinine_Level      5000 non-null   float64
 8   Donor_BMI                   5000 non-null   float64
 9   Donor_Medical_History       3767 non-null   object 
 10  Recipient_ID                5000 non-null   object 
 11  Recipient_Age               5000 non-null   int64  
 12  Recipient_Gender            5000 non-null   object 
 13  Recipient_Blood_Type        5000 

In [8]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

In [9]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

In [10]:
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
X[categorical_cols] = X[categorical_cols]
X.columns = df.drop(["Match_Label"], axis=1).columns

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
from imblearn.over_sampling import SMOTE
print("--- Original Training Set Class Distribution ---")
print(y_train.value_counts())
print("-" * 50)

print("\nApplying SMOTE to the training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\n--- Resampled Training Set Class Distribution (SMOTE) ---")
print(y_train_resampled.value_counts())
print("-" * 50)

--- Original Training Set Class Distribution ---
Match_Label
1    2000
0    2000
Name: count, dtype: int64
--------------------------------------------------

Applying SMOTE to the training data...

--- Resampled Training Set Class Distribution (SMOTE) ---
Match_Label
1    2000
0    2000
Name: count, dtype: int64
--------------------------------------------------


In [13]:
clf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [14]:
clf.feature_importances_

array([1.96118023e-01, 2.66630066e-03, 4.77463172e-04, 3.00976606e-03,
       1.04881687e-03, 9.90557117e-04, 1.40466486e-03, 3.62740231e-03,
       3.75186623e-03, 9.75688314e-04, 1.96227031e-01, 3.18852962e-03,
       5.55783610e-04, 4.37908111e-03, 1.10415743e-03, 9.64408040e-04,
       1.03969964e-03, 4.47216793e-03, 3.37627155e-03, 7.87391794e-04,
       1.14498989e-03, 5.68689940e-01])

In [15]:
y_pred = clf.predict(X_test)

In [16]:
acc = accuracy_score(y_test, y_pred)
print(f"Final Model Accuracy: {acc*100:.2f}%")

Final Model Accuracy: 99.90%


In [17]:
class_repo = classification_report(y_test, y_pred)
class_repo

'              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00       500\n           1       1.00      1.00      1.00       500\n\n    accuracy                           1.00      1000\n   macro avg       1.00      1.00      1.00      1000\nweighted avg       1.00      1.00      1.00      1000\n'

In [18]:
import pickle
try:
    with open('kidney_transplant_model.pkl', 'wb') as model_file:
        pickle.dump(clf, model_file)
    print("Model saved as kidney_transplant_model.pkl")

    with open('scaler.pkl', 'wb') as scaler_file:
        pickle.dump(scaler, scaler_file)
    print("Scaler saved as scaler.pkl")

    with open('label_encoders.pkl', 'wb') as le_file:
        pickle.dump(label_encoders, le_file)
    print("Label encoders saved as label_encoders.pkl")

except Exception as e:
    print(f"Error saving files: {e}")

Model saved as kidney_transplant_model.pkl
Scaler saved as scaler.pkl
Label encoders saved as label_encoders.pkl


In [19]:
def predict_match(sample_row):
    row = sample_row.copy()
    pred = clf.predict([row])[0]
    return "Matched" if pred == 1 else "Unmatched"

In [20]:
example = X_test.iloc[9].copy()
prediction = predict_match(example)
print("\ Example Prediction:", prediction)

\ Example Prediction: Unmatched


<>:3: SyntaxWarning: invalid escape sequence '\ '
<>:3: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_9226/734438044.py:3: SyntaxWarning: invalid escape sequence '\ '
  print("\ Example Prediction:", prediction)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [21]:
# Combine all necessary components into a single dictionary
pipeline_components = {
    'model': clf,
    'scaler': scaler,
    'label_encoders': label_encoders
}

# Save the combined pipeline components to a single pickle file
try:
    with open('kidney_transplant_pipeline.pkl', 'wb') as f:
        pickle.dump(pipeline_components, f)
    print("Full pipeline components saved as kidney_transplant_pipeline.pkl")
except Exception as e:
    print(f"Error saving full pipeline: {e}")


Full pipeline components saved as kidney_transplant_pipeline.pkl


In [22]:
import pickle
filename = 'kidney_transplant.sav'
pickle.dump(clf,open(filename,'wb'))
loaded_model = pickle.load(open('kidney_transplant.sav','rb'))

In [23]:
def predict_match(sample_row):
    row = sample_row.copy()
    pred = loaded_model.predict([row])[0]
    return "Matched" if pred == 1 else "Unmatched"

In [24]:
example = X_test.iloc[7].copy()
prediction = predict_match(example)
print("\nExample Prediction:", prediction)


Example Prediction: Matched


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [25]:
from importlib.metadata import version
print(version('scikit-learn'))

1.8.0
